In [ ]:
# ============ FEED IMAGES & LABELS TO MODEL - COMPLETE GUIDE ============
"""
How to load images and labels, then feed to model for training
"""

print("="*80)
print("FEEDING IMAGES & LABELS TO MODEL")
print("="*80)

guide = """
╔════════════════════════════════════════════════════════════════════════════╗
║                    HOW DATA FLOWS INTO THE MODEL                          ║
╚════════════════════════════════════════════════════════════════════════════╝

STEP 1: PREPARE FILES
═════════════════════════════════════════════════════════════════════════════
You have:
   ├── train/labels.txt      (format: image_name.png\ttext)
   ├── val/labels.txt
   └── test/labels.txt

Example labels.txt content:
   page_001_line_0001.png	The quick brown fox
   page_001_line_0002.png	jumps over the lazy dog
   ...

STEP 2: LOAD IMAGES & LABELS
═════════════════════════════════════════════════════════════════════════════
Use PyTorch DataLoader to load:
   ├── Image data (PNG file → numpy array → tensor)
   ├── Label text (string → tokenize → tensor)
   └── Batch them together

STEP 3: FEED TO MODEL
═════════════════════════════════════════════════════════════════════════════
Model receives:
   ├── Input:  Image tensor (B, C, H, W)
   ├── Target: Text tensor (B, max_len)
   └── Output: Predicted text

STEP 4: TRAINING LOOP
═════════════════════════════════════════════════════════════════════════════
For each batch:
   1. Feed images to model
   2. Model processes and outputs prediction
   3. Compare prediction with true label
   4. Calculate loss
   5. Backprop to update weights
   6. Repeat for all batches

═════════════════════════════════════════════════════════════════════════════
"""

print(guide)

# ============ ACTUAL IMPLEMENTATION ============

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from pathlib import Path
import pandas as pd

print("\n" + "="*80)
print("IMPLEMENTATION: CUSTOM DATASET CLASS")
print("="*80)

# ```python
# ✅ Step 1: Create Custom Dataset Class
class OCRDataset(Dataset):
    """
    Custom PyTorch Dataset for OCR
    Loads images and their corresponding text labels
    """
    
    def __init__(self, labels_file, images_dir, max_text_len=100):
        """
        Args:
            labels_file: Path to labels.txt (format: image_name.png\ttext)
            images_dir: Path to folder containing images
            max_text_len: Maximum text length (for padding)
        """
        self.images_dir = Path(images_dir)
        self.max_text_len = max_text_len
        self.samples = []
        
        # Read labels file
        print(f"   📂 Reading labels from: {labels_file}")
        
        with open(labels_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    image_name, text = parts
                    image_path = self.images_dir / image_name
                    
                    # Verify image exists
                    if image_path.exists():
                        self.samples.append({
                            'image': image_path,
                            'text': text
                        })
        
        print(f"   ✓ Loaded {len(self.samples)} samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        """
        Get single sample
        Returns: (image_tensor, text_tensor)
        """
        sample = self.samples[idx]
        
        # ✅ LOAD IMAGE
        image = cv2.imread(str(sample['image']))
        if image is None:
            print(f"   ⚠️ Cannot read: {sample['image']}")
            return None
        
        # Convert BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Normalize
        image = image.astype('float32') / 255.0
        
        # Convert to tensor (C, H, W)
        image = torch.from_numpy(image).permute(2, 0, 1)
        
        # ✅ ENCODE TEXT
        text = sample['text']
        
        # Simple encoding: convert characters to ASCII values
        text_encoded = [ord(c) for c in text]
        
        # Pad or truncate to max length
        if len(text_encoded) < self.max_text_len:
            text_encoded = text_encoded + [0] * (self.max_text_len - len(text_encoded))
        else:
            text_encoded = text_encoded[:self.max_text_len]
        
        # Convert to tensor
        text_tensor = torch.tensor(text_encoded, dtype=torch.long)
        
        return {
            'image': image,
            'text': text_tensor,
            'text_str': text
        }


# ============ SIMPLE USAGE EXAMPLE ============

print("\n" + "="*80)
print("USAGE EXAMPLE: CREATE DATALOADER")
print("="*80)

# Create dataset
print("\n✅ Creating training dataset...")

train_dataset = OCRDataset(
    labels_file='/content/drive/MyDrive/Training_Dataset/train/labels.txt',
    images_dir='/content/drive/MyDrive/Training_Dataset/train',
    max_text_len=100
)

# Create DataLoader
print("\n✅ Creating DataLoader...")

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f"   ✓ Total batches: {len(train_loader)}")
print(f"   ✓ Batch size: 32")
print(f"   ✓ Total training samples: {len(train_dataset)}")

# ============ HOW TO USE IN TRAINING LOOP ============

print("\n" + "="*80)
print("TRAINING LOOP: HOW TO FEED DATA")
print("="*80)

training_loop_example = """
# THIS IS PSEUDOCODE - shows how to use the DataLoader

for epoch in range(num_epochs):
    for batch_idx, batch in enumerate(train_loader):
        # ✅ GET DATA FROM LOADER
        images = batch['image'].to(device)      # Shape: (batch_size, 3, height, width)
        texts = batch['text'].to(device)        # Shape: (batch_size, max_text_len)
        
        # ✅ FORWARD PASS: Feed images to model
        outputs = model(images)                 # Model processes images
                                                # Output shape: (batch_size, max_text_len, vocab_size)
        
        # ✅ CALCULATE LOSS: Compare output with true label
        loss = criterion(outputs, texts)        # Compare prediction vs ground truth
        
        # ✅ BACKWARD PASS: Update weights
        optimizer.zero_grad()                   # Clear gradients
        loss.backward()                         # Calculate gradients
        optimizer.step()                        # Update weights
        
        # ✅ PRINT PROGRESS
        if batch_idx % 100 == 0:
            print(f"Batch {batch_idx}/{len(train_loader)} - Loss: {loss:.4f}")
"""

print(training_loop_example)

In [ ]:
# ============ COMPLETE WORKING EXAMPLE ============
"""
Full working example with model training
"""

print("\n" + "="*80)
print("COMPLETE WORKING EXAMPLE")
print("="*80)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from pathlib import Path
import time

# ✅ 1. CUSTOM DATASET CLASS
class OCRDataset(Dataset):
    def __init__(self, labels_file, images_dir, max_text_len=100):
        self.images_dir = Path(images_dir)
        self.max_text_len = max_text_len
        self.samples = []
        
        with open(labels_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    image_name, text = parts
                    image_path = self.images_dir / image_name
                    if image_path.exists():
                        self.samples.append({'image': image_path, 'text': text})
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Load and process image
        image = cv2.imread(str(sample['image']))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image.astype('float32') / 255.0
        image = torch.from_numpy(image).permute(2, 0, 1)
        
        # Encode text
        text = sample['text']
        text_encoded = [ord(c) for c in text]
        
        if len(text_encoded) < self.max_text_len:
            text_encoded += [0] * (self.max_text_len - len(text_encoded))
        else:
            text_encoded = text_encoded[:self.max_text_len]
        
        text_tensor = torch.tensor(text_encoded, dtype=torch.long)
        
        return {
            'image': image,
            'text': text_tensor,
            'text_str': text
        }

# ✅ 2. SIMPLE OCR MODEL
class SimpleOCRModel(nn.Module):
    def __init__(self, vocab_size=256, max_text_len=100):
        super().__init__()
        
        # CNN for feature extraction
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        # RNN for sequence
        self.rnn = nn.LSTM(input_size=128, hidden_size=256, 
                          num_layers=2, batch_first=True, bidirectional=True)
        
        # Output layer
        self.fc = nn.Linear(512, vocab_size)
    
    def forward(self, x):
        # CNN
        x = self.cnn(x)
        
        # Reshape for RNN
        b, c, h, w = x.size()
        x = x.view(b, c, -1).permute(0, 2, 1)
        
        # RNN
        x, _ = self.rnn(x)
        
        # Output
        x = self.fc(x)
        
        return x

# ✅ 3. TRAINING FUNCTION
def train_model(train_loader, val_loader, epochs=10):
    """
    Complete training function
    Shows how to feed data and train
    """
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n✅ Using device: {device}")
    
    # Initialize model
    model = SimpleOCRModel()
    model.to(device)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    print(f"\n✅ Model initialized")
    print(f"   Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Training loop
    print(f"\n{'='*80}")
    print(f"STARTING TRAINING")
    print(f"{'='*80}\n")
    
    for epoch in range(epochs):
        print(f"\n{'─'*80}")
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"{'─'*80}\n")
        
        # Training
        model.train()
        train_loss = 0
        
        for batch_idx, batch in enumerate(train_loader):
            # ✅ FEED DATA TO GPU
            images = batch['image'].to(device)
            texts = batch['text'].to(device)
            
            # ✅ FORWARD PASS
            outputs = model(images)
            
            # ✅ CALCULATE LOSS
            # Reshape for loss calculation
            outputs_flat = outputs.view(-1, outputs.size(-1))
            texts_flat = texts.view(-1)
            
            loss = criterion(outputs_flat, texts_flat)
            
            # ✅ BACKWARD PASS
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            # Progress
            if batch_idx % 10 == 0:
                print(f"Batch [{batch_idx+1:4d}/{len(train_loader)}] "
                      f"Loss: {loss.item():.4f}")
        
        avg_train_loss = train_loss / len(train_loader)
        print(f"\n✓ Training Loss: {avg_train_loss:.4f}")
        
        # Validation
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(device)
                texts = batch['text'].to(device)
                
                outputs = model(images)
                outputs_flat = outputs.view(-1, outputs.size(-1))
                texts_flat = texts.view(-1)
                
                loss = criterion(outputs_flat, texts_flat)
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        print(f"✓ Validation Loss: {avg_val_loss:.4f}")
        
        # Save checkpoint
        if (epoch + 1) % 5 == 0:
            checkpoint_path = f'/content/drive/MyDrive/Models/checkpoint_epoch_{epoch+1}.pt'
            torch.save(model.state_dict(), checkpoint_path)
            print(f"\n✓ Checkpoint saved: checkpoint_epoch_{epoch+1}.pt")
    
    print(f"\n{'='*80}")
    print(f"✅ TRAINING COMPLETE!")
    print(f"{'='*80}")
    
    return model

# ✅ 4. CREATE DATASETS AND DATALOADERS
print("\n✅ Creating datasets...")

train_dataset = OCRDataset(
    labels_file='/content/drive/MyDrive/Training_Dataset/train/labels.txt',
    images_dir='/content/drive/MyDrive/Training_Dataset/train',
    max_text_len=100
)

val_dataset = OCRDataset(
    labels_file='/content/drive/MyDrive/Training_Dataset/val/labels.txt',
    images_dir='/content/drive/MyDrive/Training_Dataset/val',
    max_text_len=100
)

print(f"\n   ✓ Train samples: {len(train_dataset)}")
print(f"   ✓ Val samples: {len(val_dataset)}")

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n   ✓ Train batches: {len(train_loader)}")
print(f"   ✓ Val batches: {len(val_loader)}")

# ✅ 5. TRAIN MODEL
print("\n✅ Starting training...")

model = train_model(train_loader, val_loader, epochs=5)

# ✅ 6. SAVE FINAL MODEL
print("\n✅ Saving final model...")

model_path = '/content/drive/MyDrive/Models/final_model.pt'
torch.save(model.state_dict(), model_path)

print(f"   ✓ Model saved: {model_path}")
print(f"\n✅ COMPLETE!")